In [1]:
import json
import yaml
from mstrio.connection import Connection

from mstr_robotics.mstr_classes import mstr_global
from mstr_robotics._helper import msic
from mstr_robotics.redis_db import  redis_bi_analysis,redis_mstr_json
from mstr_robotics._connectors import mstr_api
from mstr_robotics.prepare_AI_data import export_mstr_md


i_msic=msic()
i_mstr_global=mstr_global()
i_mstr_api=mstr_api()
i_redis_mstr_json=redis_mstr_json()
i_export_mstr_md=export_mstr_md()


with open('..\\config\\mstr_redis_y.yml', 'r') as openfile:
    mstr_redis_y = yaml.safe_load(openfile)


In [2]:
redis_con_d=mstr_redis_y["redis_env_d"]["redis_dev"]
project_prefix=mstr_redis_y["project_prefix"]
prefix_map=mstr_redis_y["prefix_map"]
searches_used_in_prp_d_l=mstr_redis_y["searches_used_in_prp_d_l"]


## Connect to MSTR & Redis

In [3]:
with open('..\\config\\user_d.json', 'r') as openfile:
    user_d = json.load(openfile)
conn_params =  user_d["conn_params"]
conn = Connection(**conn_params)


i_redis_bi_analysis = redis_bi_analysis( 
    host=redis_con_d["host"],
    port=redis_con_d["port"],
    password=redis_con_d["password"],
    username=redis_con_d["username"],
    decode_responses=redis_con_d["decode_responses"]
)


Connection to Strategy One Intelligence Server has been established.
No project selected.


## Save objects to redis

In [ ]:
load_type_fg='daily_search_update'
load_type_fg='full_load'
load_type_fg='shortcut_folder'
load_type_fg='single_objects'

### Daily Search Update

In [4]:
if load_type_fg=="daily_search_update":
    err_d_l=[]
    for pre in project_prefix:
        env_prefix=project_prefix[pre]
        conn.select_project(pre)
        search_result = i_mstr_api.run_mstr_search(conn=conn,
                            search_id="96648F2B492150A6AA27DDB3744E32B4")
    
        if search_result["totalItems"] > 0:
            all_obj_d_l=search_result["result"]
    
            err_d_l.append(i_redis_mstr_json.save_obj_json_to_redis(i_redis_bi_analysis=i_redis_bi_analysis                                                    
                                                            ,conn=conn
                                                            ,prefix_map=prefix_map
                                                            , all_obj_d_l=all_obj_d_l
                                                            , env_prefix=env_prefix)
                            )

{'totalItems': 1, 'result': [{'name': 'Customer Detail report (Dashboard)_C060EF484E1352B26D945CBAF191F2B8_sess_7417833397931741184_rep_job4333', 'id': 'CF22536A48F5CEB8C38EC7BB3689884E', 'type': 3, 'description': 'string', 'subtype': 768, 'dateCreated': '2026-06-20T19:43:58.000+0000', 'dateModified': '2026-06-20T19:44:11.000+0000', 'version': 'B70704FA41A21D315E63F29BDF0590A1', 'acg': 255, 'owner': {'name': 'Administrator', 'id': '54F3D26011D2896560009A8E67019608', 'expired': False}, 'extType': 1, 'viewMedia': 134217728, 'certifiedInfo': {'certified': False}, 'projectId': 'B7CA92F04B9FAE8D941C3E9B7E0CD754', 'managed': False}]}
Uploaded object definition for CF22536A48F5CEB8C38EC7BB3689884E
[{'name': 'Customer Detail report (Dashboard)_C060EF484E1352B26D945CBAF191F2B8_sess_7417833397931741184_rep_job4333', 'id': 'CF22536A48F5CEB8C38EC7BB3689884E', 'type': 3, 'description': 'string', 'subtype': 768, 'dateCreated': '2026-06-20T19:43:58.000+0000', 'dateModified': '2026-06-20T19:44:11.000+

[{'name': 'F_TUTORIAL_REGION_TARGETS',
  'id': '1C0A3D964E21D5D517D915958D284694',
  'type': 15,
  'subtype': 3840,
  'dateCreated': '2012-01-27T12:00:26.000+0000',
  'dateModified': '2026-06-21T06:38:41.000+0000',
  'version': '215230EB4F7089CD0261C1AD12D621AD',
  'acg': 255,
  'owner': {'name': 'Administrator',
   'id': '54F3D26011D2896560009A8E67019608',
   'expired': False},
  'extType': 1,
  'projectId': '74F9B2164869627AF637FDA5D4A121B3',
  'managed': False},
 {'name': 'region_id',
  'id': '8D67915D11D3E4981000E787EC6DE8A4',
  'type': 26,
  'abbreviation': 'region_id',
  'subtype': 6656,
  'dateCreated': '2001-01-02T20:48:31.000+0000',
  'dateModified': '2026-06-21T06:38:41.000+0000',
  'version': '1A429F7C5E43DE2F9CDE63972CAD90D4',
  'acg': 255,
  'owner': {'name': 'Administrator',
   'id': '54F3D26011D2896560009A8E67019608',
   'expired': False},
  'extType': 1,
  'projectId': '74F9B2164869627AF637FDA5D4A121B3',
  'managed': False},
 {'name': 'region_name',
  'id': '8D67927611D

### Full load

In [5]:
if daily_search_update=="full_load":
    # Initialize Redis connection
    env_prefix="mstr_test"
    i_redis_bi_analysis.emergency_flush_db(confirm_phrase='FLUSH_ALL_DATA')
    
    for pre in project_prefix:
        env_prefix=project_prefix[pre]
        conn.select_project(pre)
        all_obj_d_l=i_export_mstr_md.read_out_prj_by_type(conn)
    
        err_d_l=i_redis_mstr_json.save_obj_json_to_redis(i_redis_bi_analysis=i_redis_bi_analysis
                                                        ,conn=conn
                                                        ,prefix_map=prefix_map
                                                        , all_obj_d_l=all_obj_d_l
                                                        , env_prefix=env_prefix)
    
        i_redis_mstr_json.run_searches_redis(conn=conn
                                            ,i_redis_bi_analysis=i_redis_bi_analysis
                                            ,prefix_map=prefix_map
                                            ,env_prefix=env_prefix
                                            ,searches_used_in_prp_d_l=searches_used_in_prp_d_l)
      

JJJJJ
168
923
715
4
206
118
36
11
74
0
0
0
0
0
0
8
5
2
91
469
8
25
Uploaded object definition for 8FE9684E11D5287510006389EBC3E6A4
Uploaded object definition for 04373BB64B0177EA2A662E9AF58378F5
Uploaded object definition for 5EEF50EB42EEEBA8CBCC20B42FC5162D
Uploaded object definition for 613437284BB4A2849B52F4A31D80D418
Uploaded object definition for C66FB1D911D3EB0CC000B4B2D86C964F
Uploaded object definition for 085CBF314703A71E926EA187C2002969
Uploaded object definition for 8045236C492EB6A00E795996F0302F18
Uploaded object definition for 5329490D46085BD3D49CE3AA84EC3100
Uploaded object definition for E03EB3FC49A3EF156852AB9B543A0C1D
Uploaded object definition for 1A8DEEDA4913BE24385997B55F9E526D
Uploaded object definition for 2A08F3C946A1255CE2EBF9AF6B12EFE9
Uploaded object definition for 8827905B11D3EB22C000B4B2D86C964F
Uploaded object definition for 8827904B11D3EB22C000B4B2D86C964F
Uploaded object definition for 8827902011D3EB22C000B4B2D86C964F
Uploaded object definition for 882790

### Shortcut Folder

In [6]:
if daily_search_update=="shortcut_folder":
    folder_id="B70AF11248C00F7D4339079310F8B3F1"
    project_id="B7CA92F04B9FAE8D941C3E9B7E0CD754"
    conn.select_project(project_id)
    all_obj_d_l=i_mstr_global.get_obj_from_sh_fold(conn,folder_id=folder_id)

Folder object named: 'testObjs' with ID: 'B70AF11248C00F7D4339079310F8B3F1'


### Single Objects

In [7]:
if daily_search_update=="shortcut_folder":
    all_obj_d_l=[]
    all_obj_d_l.append({"id":"C58A830B4DEB35BF601D119134952B83","type":"3","subtype":"768"})


In [8]:
redis_mstr_d={"project_prefix":{"B7CA92F04B9FAE8D941C3E9B7E0CD754":"mstr_dev",
                                "74F9B2164869627AF637FDA5D4A121B3":"mstr_test"}}

env_prefix=redis_mstr_d["project_prefix"][project_id]

conn.select_project(project_id)
err_d_l=i_redis_mstr_json.save_obj_json_to_redis(i_redis_bi_analysis=i_redis_bi_analysis                                                    
                                                    ,conn=conn
                                                    ,prefix_map=prefix_map
                                                    , all_obj_d_l=all_obj_d_l
                                                    , env_prefix=env_prefix)

err_d_l

Uploaded object definition for C58A830B4DEB35BF601D119134952B83


[]